In [1]:
# Nikolay Vorontsov, 20.10.2024
# This notebook is for opening and viewing the evaluation files
# for Mu-SHROOM shared task (SemEval 2025 Task 3), and some further experiment with them.
#
# references:
# https://chatgpt.com/share/6720c6e7-44ac-800b-bce9-aeafe43c5aa5
# https://chatgpt.com/share/6720c718-9748-800b-8fae-f2c22eb8540c
#

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json
import pprint
import textwrap
import random
import re

In [1]:
#CHECK THE PATH ACCORDING TO YOUR OWN DRIVE
path_to_data = "/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/data/"

In [ ]:
##Set a file id number: 1 -- 50 for English
id = 10

In [ ]:
# Define the target ID
target_id = "val-en-" + str(id)

# Define target_output string
target_output = ""

# Open and read the .jsonl file
with open(path_to_data + "/val/mushroom.en-val.v2.jsonl", "r", encoding="utf-8") as file:
    for line in file:
        # Parse each line as a JSON object
        data = json.loads(line)

        # Check if the current object's ID matches the target ID
        if data["id"] == target_id:
            """
            # Print the JSON object structure
            for x in data:
              print(x)
            print("\n")
            """
            # Print for particular id:
            for key, value in data.items():
              if key == "model_output_text":
                print(f"{key}: {value}\n")
                # Export to target_output:
                target_output = value
                # Wrapping the output text to a specific width
                wrapped_output = textwrap.fill(data["model_output_text"], width=50)
                # Print the wrapped output
                print("WRAPPED OUTPUT")
                print(wrapped_output)
                print("\n")
              else:
                print(f"{key}: {value}\n")

            ### NOTE FOR FUTURE: may need to use json.dumps / json.dump
            json_string = json.dumps(data, ensure_ascii=False, indent=4)
            print(json_string)

            break  # Exit the loop once the desired entry is found

id: val-en-30

lang: EN

model_input: What was Valrico, Florida known as before the Civil War?

model_output_text:  Before the Civil War, Valrico, Florida was not a well-known or established settlement. The area around Valrico was originally inhabited by various Native American tribes, including the Seminoles. The first European settlers arrived in the mid-1800s, and the community began to develop after the completion of the Tampa-Hillsborough Railroad in 1885. Therefore, there is no recorded history of Valrico before the Civil War.

WRAPPED OUTPUT
 Before the Civil War, Valrico, Florida was not a
well-known or established settlement. The area
around Valrico was originally inhabited by various
Native American tribes, including the Seminoles.
The first European settlers arrived in the
mid-1800s, and the community began to develop
after the completion of the Tampa-Hillsborough
Railroad in 1885. Therefore, there is no recorded
history of Valrico before the Civil War.


model_id: TheBloke/

In [ ]:
## Experiment with a particular output
## May use training data here...

print(textwrap.fill(target_output, width=80))


 Before the Civil War, Valrico, Florida was not a well-known or established
settlement. The area around Valrico was originally inhabited by various Native
American tribes, including the Seminoles. The first European settlers arrived in
the mid-1800s, and the community began to develop after the completion of the
Tampa-Hillsborough Railroad in 1885. Therefore, there is no recorded history of
Valrico before the Civil War.


In [ ]:
### Note: CHECK validation scripts for various language -- this is how the data was produced

In [ ]:
## Filter the validation set: remove the soft labels, hard labels

# Define paths for input and output files
path_to_data = "/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/data/"
input_file_path = path_to_data + "val/mushroom.en-val.v2.jsonl"
output_file_path = path_to_data + "val/mushroom.en-val_filtered.jsonl"

# Specify the required model ID and language code
model_id = "togethercomputer/Pythia-Chat-Base-7B"
lang_code = "EN"

# Open the input file and create a new output file
with open(input_file_path, 'r', encoding='utf-8') as infile, open(output_file_path, 'w', encoding='utf-8') as outfile:
    for line in infile:
        # Parse each line as a JSON object
        data = json.loads(line)

        # Filter out the required fields
        filtered_entry = {
            "lang": lang_code,
            "model_id": model_id,
            "model_input": data.get("model_input", ""),
            "model_output_text": data.get("model_output_text", ""),
            "model_output_logits": data.get("model_output_logits", []),
            "model_output_tokens": data.get("model_output_tokens", [])
        }

        # Write the filtered entry to the output file in JSONL format
        json.dump(filtered_entry, outfile, ensure_ascii=False)
        outfile.write('\n')  # Ensure each entry is on a new line

print(f"Filtered data saved to {output_file_path}")


In [ ]:
## DETECT SPANS
## 1. implement simple detection

## 2. apply other tools

In [ ]:
# Sample text
text = target_output

**SPAN_DETECTION FUNCTION -- this is where all the work goes in**

In [ ]:
# Example spans (start, end) tuples. Soft span labels can in theory overlap. Hard labels should not overlap.
spans = [
    (0, 51),
    (89, 95),
    (100, 110)
]

### HERE SHOULD BE A PROPER SPAN_DETECTOR ::

# Function to detect spans in text
def spand_detector(text):
  # Initialize a list to store detected spans
  super_spans = []
  # Define a regular expression pattern to match numeric characters
  pattern = r'\d+.*$'
  # Find all matches of the pattern in the text
  matches = re.finditer(pattern, text)
  # Iterate through the matches and extract the spans
  for match in matches:
    super_spans.append((match.start(), match.end()))

  return super_spans
print(spand_detector(text))
print(len(text))
## ASSIGN Detected super_spans to spans
spans = spand_detector(text)

[(244, 423)]
423


In [ ]:
# Function to extract spans from text
def extract_spans(text, spans):
    # Initialize a list to store extracted spans
    extracted_spans = []
    for start, end in spans:
        span_text = text[start:end]  # Extract text using the start and end indices
        extracted_spans.append(span_text)  # Append to the result list
    return extracted_spans

In [ ]:
## Pretty_printing of detected spans

# Extracting the spans
detected_spans = extract_spans(text, spans)

# Printing the detected spans
for i, span in enumerate(detected_spans):
    print(f"Span {i + 1}: {span}")


Span 1: 1800s, and the community began to develop after the completion of the Tampa-Hillsborough Railroad in 1885. Therefore, there is no recorded history of Valrico before the Civil War.


In [ ]:
## REPEAT ON A LARGER FILE, with mulptiple texts.
## Use training set here.

In [ ]:
## Proccess the training dataset:

file_path = path_to_data + "/train/mushroom.en-train_nolabel.v1.jsonl"

# Initialize a counter
line_count = 0

# Open the file and count the lines
with open(file_path, 'r', encoding='utf-8') as file:
    for line in file:
        line_count += 1

print(f"Total number of data points: {line_count}")

Total number of data points: 809


In [ ]:
random_number = random.randint(0, int(line_count))
target_id = random_number
print(target_id)

484


In [ ]:
## Print random/targeted text_output and input

# Initialise a counter
line_count = 0

# Initialise new data
new_data = ""

with open(file_path, 'r', encoding='utf-8') as file:
    for line in file:
        training_data = json.loads(line)
        if line_count == random_number:  ## or target in en-train from 0 to 808
            print("line_number:", line_count)
            print("model_input:", training_data["model_input"])
            print("model_output_text:", training_data["model_output_text"])
            new_data = training_data["model_output_text"]
            break
        line_count += 1



line_number: 484
model_input: When did Jef Raskin die?
model_output_text: Jef's death date is unknown.


In [ ]:
extract_spans(new_data, spans)

NameError: name 'extract_spans' is not defined

In [ ]:
##Print the whole datapoint if needed
training_data

{'lang': 'EN',
 'model_id': 'togethercomputer/Pythia-Chat-Base-7B',
 'model_input': 'Who was the father of Carl Löwenhielm?',
 'model_output_text': ' Carl Gustaf Bernhard Lönneberga was born in Stockholm, Sweden, on March 24, 1754, the son of Gustavus Adolphus Lønneberg and Karin Elisabeth Eriksdotter. ',
 'model_output_logits': [-2.0140974522,
  12.7265701294,
  6.6418390274,
  0.146374464,
  3.1201014519,
  1.2584956884,
  -2.4574751854,
  5.6701946259,
  -2.8140125275,
  7.0877485275,
  -2.7370963097,
  8.1359539032,
  6.0415964127,
  7.85100317,
  6.838906765,
  3.6724581718,
  8.6903839111,
  3.7015023232,
  9.9995450974,
  6.5371389389,
  7.2181353569,
  3.4407482147,
  5.4814519882,
  -1.9529628754,
  4.8478927612,
  7.232509613,
  8.8106708527,
  6.5563640594,
  9.8751163483,
  -1.6160643101,
  6.4435138702,
  2.3907122612,
  1.5956583023,
  7.1813936234,
  0.5025379658,
  11.58111763,
  6.3621439934,
  -1.8913232088,
  0.8921307325,
  4.8449478149,
  2.1555838585,
  -1.5500822

In [ ]:
## CONTINUE processing data, loop over all the training set / validation set. Remember about overfitting!

In [ ]:
## EXPORT INTO JSONL FILE

In [ ]:
## EVALUATION, check random_guess , scorer.py , baseline(not realised)